### Created a Simple RAG pipeline

In [6]:
!pip install langchain langchain_community langchain_google_genai langchain_text_splitters
!pip install "unstructured[all-docs]"


In [7]:
import os

from google.colab import userdata
gem = userdata.get('gemini')

os.environ["GOOGLE_API_KEY"] = gem

In [8]:
# 1. step document loader

from langchain_community.document_loaders import PyPDFLoader

doc = PyPDFLoader("/content/HR-Policies-Manuals.pdf")
document = doc.load()

In [9]:
len(document) # Document contain 24 pages (document_object)

24

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
# 2. text_splitter --> Chunking
rts = RecursiveCharacterTextSplitter(separators= ["\n\n","\n"," "],
                               chunk_size = 500, chunk_overlap=10)

# Seperated by "\n\n":Paregraph , "\n":Sentence, " ":Space ,"":word
#chunk_overlap = 10 means:
# When splitting text into chunks, the next chunk repeats the last 10 characters of the previous chunk.

In [12]:
chunks = rts.split_documents(document)

In [13]:
len(chunks)

108

In [14]:
# step 3 and 4 --> Embedding and storing
# Use Huggingface Embedding model

!pip install transformers
!pip install sentence_transformers #Many of the huggingface embedding models are present into this library
!pip install langchain-huggingface

In [16]:
!pip install -U langchain langchain-community chromadb

In [15]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
vec_db = Chroma.from_documents(documents =chunks, embedding= emb, persist_directory="db")
vec_db.persist()

In [21]:
# Try to make database Retrieval
retriever = vec_db.as_retriever(search_kwargs={"k":3}) #This is going to wrap vector database as a runnable object called retriever

In [22]:
retriever.invoke("Hi how are you")

[Document(metadata={'source': '/content/HR-Policies-Manuals.pdf', 'author': 'hr', 'creationdate': '2020-08-26T06:56:00+00:00', 'page': 3, 'creator': 'Microsoft® Word 2016', 'producer': 'www.ilovepdf.com', 'page_label': '4', 'total_pages': 24, 'moddate': '2020-08-26T06:56:00+00:00'}, page_content='GREAT REWARD AND OPPORTUNITY OF WORK WITH TEAM AND HELPFUL ATMOSPHERE  \n \nThis policy covers the vacant position on PAN India basis across the functions, department, level, \ngrade and hierarchy. The following steps are to be followed to hire the candidate  \n \na) The proper details will be given to the HR Department related to the candidate job description, \nage, gender, salary package, no of subordinates, qualifications and experience.'),
 Document(metadata={'page_label': '4', 'creator': 'Microsoft® Word 2016', 'author': 'hr', 'creationdate': '2020-08-26T06:56:00+00:00', 'total_pages': 24, 'producer': 'www.ilovepdf.com', 'source': '/content/HR-Policies-Manuals.pdf', 'page': 3, 'moddate':

In [23]:
def f1(data):
  l = []

  for y in data: #loop through documents
    l.append(y.page_content)
  return "\n\n".join(l)

In [24]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, RunnableSequence

In [26]:
runn2= RunnableLambda(f1)

In [32]:
RunnableSequence(retriever,runn2).invoke("Hi,how are you") #This give entire context

'GREAT REWARD AND OPPORTUNITY OF WORK WITH TEAM AND HELPFUL ATMOSPHERE  \n \nThis policy covers the vacant position on PAN India basis across the functions, department, level, \ngrade and hierarchy. The following steps are to be followed to hire the candidate  \n \na) The proper details will be given to the HR Department related to the candidate job description, \nage, gender, salary package, no of subordinates, qualifications and experience.\n\nGREAT REWARD AND OPPORTUNITY OF WORK WITH TEAM AND HELPFUL ATMOSPHERE  \n \nThis policy covers the vacant position on PAN India basis across the functions, department, level, \ngrade and hierarchy. The following steps are to be followed to hire the candidate  \n \na) The proper details will be given to the HR Department related to the candidate job description, \nage, gender, salary package, no of subordinates, qualifications and experience.\n\nh) “Posted City” means the work place where the employee will be posted at the time of joining.  \n \

In [31]:
chain1= RunnableSequence(retriever,runn2)

In [29]:
RunnableParallel({"context":chain1, "question": RunnablePassthrough()}).invoke("Hi,how are you")


{'context': 'GREAT REWARD AND OPPORTUNITY OF WORK WITH TEAM AND HELPFUL ATMOSPHERE  \n \nThis policy covers the vacant position on PAN India basis across the functions, department, level, \ngrade and hierarchy. The following steps are to be followed to hire the candidate  \n \na) The proper details will be given to the HR Department related to the candidate job description, \nage, gender, salary package, no of subordinates, qualifications and experience.\n\nGREAT REWARD AND OPPORTUNITY OF WORK WITH TEAM AND HELPFUL ATMOSPHERE  \n \nThis policy covers the vacant position on PAN India basis across the functions, department, level, \ngrade and hierarchy. The following steps are to be followed to hire the candidate  \n \na) The proper details will be given to the HR Department related to the candidate job description, \nage, gender, salary package, no of subordinates, qualifications and experience.\n\nh) “Posted City” means the work place where the employee will be posted at the time of jo

In [33]:
chain2 =RunnableParallel({"context":chain1, "question": RunnablePassthrough()})

In [35]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate

cpt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are a helpful HR assistant who is having 4 years of experience"),
                                 HumanMessagePromptTemplate.from_template("Answer the question based on the below given context and if there is no context then please return I don not know {context}\n\n{question}")])
# It is a runnable

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

In [37]:
# Runnable 2 = Set LLM Model
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash-lite")

In [39]:
# Runnable 3 = Output Parser
sto = StrOutputParser()

In [40]:
chain3 = RunnableSequence(cpt,model,sto)

In [41]:
final_RAG_Pipeline = RunnableSequence(chain2,chain3)

In [43]:
final_RAG_Pipeline.invoke("How many leaves i can take per month")

'Based on the context provided, the number of leaves you can take per month depends on your date of joining:\n\n*   **1st to 7th of every month:** 2 leaves\n*   **8th to 14th of every month:** 1.5 leaves\n*   **15th to 21st of every month:** 1 leave\n*   **22nd to 31st of every month:** 0.5 leaves\n\nThese leaves can be availed after the completion of the month.'

In [ ]:
# Disadvantage of this system is (Whatever question you give to this system it always retrieval will be called)
# So in some problem statement we want our LLM to call the [Retriever] , if you do not answer fall back and call retriever.